# Entrenamiento YOLOv8n-Pose — Giroscopio Dron (v4 · 12 keypoints)

## Estructura de keypoints
| KP | Etiqueta | Grupo | Eje |
|---|---|---|---|
| 1 | oct_tras_izq | Octógono | Pitch |
| 2 | oct_tras_der | Octógono | Pitch |
| 3 | oct_front_der | Octógono | Pitch |
| 4 | oct_front_izq | Octógono | Pitch |
| 5 | int_tras_arr | Soporte intermedio | Roll |
| 6 | int_tras_abj | Soporte intermedio | Roll |
| 7 | int_front_arr | Soporte intermedio | Roll |
| 8 | int_front_abj | Soporte intermedio | Roll |
| 9 | base_tras_izq | Base | Yaw |
| 10 | base_tras_der | Base | Yaw |
| 11 | base_front_der | Base | Yaw |
| 12 | base_front_izq | Base | Yaw |

## Estructura de carpetas en Google Drive
Todo el proyecto vive bajo una única carpeta raíz en Drive:
```
Mi unidad/
└── giroscopio_proyecto_g1/
    ├── dataset_g1/                  ← ZIP descargado de Roboflow (opción A)
    │   └── Giroscopio_G1.v3i.yolov8 Dataset 1860.zip
    └── runs/                        ← se crea automáticamente al entrenar
        └── giroscopio_v4/
            ├── weights/
            │   ├── best.pt
            │   └── last.pt
            └── (métricas, gráficas...)
```
Los archivos exportados finales (`.onnx`, `.pt`) también se guardan en `giroscopio_proyecto_g1/`.

## Instrucciones de uso
1. Menú → Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**
2. Ejecutar las celdas **en orden**, una a una
3. En la **Celda 2** elegir el modo de carga del dataset (Drive o Roboflow API)
4. Si se usa Roboflow, rellenar las credenciales en la **Celda 3**
5. Si la sesión se reinicia (timeout), volver a ejecutar desde la Celda 1 **saltando la Celda 5** (entrenamiento)


In [ ]:
# ============================================================
# CELDA 1 — Verificar GPU e instalar dependencias
# ============================================================
# Comprobamos que Colab nos ha asignado una GPU y que CUDA
# está disponible. Sin GPU el entrenamiento tardaría días.
# Instalamos ultralytics (que incluye YOLOv8) y roboflow
# (necesario solo si se usa la opción de descarga por API).

import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU:', result.stdout.strip())

!pip install ultralytics roboflow -q

import ultralytics, torch
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())

if not torch.cuda.is_available():
    print('\n⚠️  AVISO: No se detecta GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU T4')


In [ ]:
# ============================================================
# CELDA 2 — Montar Google Drive y cargar el dataset
# ============================================================
# El dataset puede cargarse de dos formas:
#
#   OPCIÓN A — Desde Drive (recomendada si ya tienes el ZIP):
#     Descarga el dataset de Roboflow en formato YOLOv8 Pose,
#     coloca el ZIP en: giroscopio_proyecto_g1/dataset_g1/
#     y pon DATASET_SOURCE = 'drive' aquí abajo.
#
#   OPCIÓN B — Desde Roboflow API (descarga en tiempo real):
#     Pon DATASET_SOURCE = 'roboflow' y rellena las
#     credenciales en la Celda 3.
#
# En ambos casos el dataset queda en /content/dataset
# para que las celdas siguientes funcionen igual.

# ── CONFIGURACIÓN ─────────────────────────────────────────
DATASET_SOURCE = 'drive'   # 'drive'  ó  'roboflow'

# Carpeta raíz del proyecto en tu Drive (NO cambiar si sigues
# la estructura recomendada en el README de arriba)
DRIVE_ROOT = '/content/drive/MyDrive/giroscopio_proyecto_g1'

# Subcarpeta donde está el ZIP descargado de Roboflow
DATASET_ZIP_DIR = f'{DRIVE_ROOT}/dataset_g1'

# Ruta local donde se extrae/descarga el dataset en Colab
DATASET_LOCAL = '/content/dataset'
# ─────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Carpeta raíz del proyecto: {DRIVE_ROOT}')

if DATASET_SOURCE == 'drive':
    # ── OPCIÓN A: extraer el ZIP desde Drive ────────────────
    import zipfile

    # Listar lo que hay en la carpeta para diagnóstico
    print('\nContenido de dataset_g1:')
    for f in os.listdir(DATASET_ZIP_DIR):
        print(f'  {repr(f)}')

    zips = [f for f in os.listdir(DATASET_ZIP_DIR) if f.lower().endswith('.zip')]
    if not zips:
        raise FileNotFoundError(
            f'No se encontró ningún .zip en {DATASET_ZIP_DIR}\n'
            f'Descarga el dataset de Roboflow (formato YOLOv8 Pose) y colócalo ahí.'
        )
    zip_path = os.path.join(DATASET_ZIP_DIR, zips[0])
    print(f'\nZIP encontrado: {zip_path}')

    os.makedirs(DATASET_LOCAL, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATASET_LOCAL)
    print(f'Dataset extraído en {DATASET_LOCAL}')

    # Normalizar estructura: si el ZIP anida una subcarpeta
    # (ej: dataset/giroscopio-1/train/) la aplanamos un nivel
    subdirs = [
        d for d in os.listdir(DATASET_LOCAL)
        if os.path.isdir(os.path.join(DATASET_LOCAL, d))
        and d not in ('train', 'valid', 'test')
    ]
    if subdirs and not os.path.exists(os.path.join(DATASET_LOCAL, 'train')):
        inner = os.path.join(DATASET_LOCAL, subdirs[0])
        print(f'Subcarpeta anidada detectada ({subdirs[0]}) → aplanando...')
        for item in os.listdir(inner):
            shutil.move(os.path.join(inner, item), DATASET_LOCAL)
        os.rmdir(inner)
        print('Estructura aplanada.')

elif DATASET_SOURCE == 'roboflow':
    # ── OPCIÓN B: se descargará en la Celda 3 ───────────────
    print('Modo Roboflow API seleccionado. Continúa con la Celda 3.')

else:
    raise ValueError(f'DATASET_SOURCE debe ser "drive" o "roboflow", no "{DATASET_SOURCE}"')

# Resumen de imágenes disponibles (solo si ya tenemos el dataset)
if DATASET_SOURCE == 'drive':
    print('\nImágenes por split:')
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(DATASET_LOCAL, split, 'images')
        n = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
        print(f'  {split}: {n} imágenes')


In [ ]:
# ============================================================
# CELDA 3 — Descargar dataset desde Roboflow (solo Opción B)
# ============================================================
# SALTAR ESTA CELDA si usas DATASET_SOURCE = 'drive' en Celda 2
#
# Para obtener los datos de conexión:
#   1. Ve a roboflow.com → tu proyecto → Versions
#   2. Haz clic en la versión → Export → Format: YOLOv8 Pose
#   3. Selecciona 'Show download code' y copia los valores

import os
DATASET_SOURCE = os.environ.get('DATASET_SOURCE', 'drive')  # hereda de Celda 2 si se ejecutó

# Rellenar solo si DATASET_SOURCE == 'roboflow'
API_KEY   = "PEGA_AQUI_TU_API_KEY"      # ← tu clave personal de Roboflow
WORKSPACE = "PEGA_AQUI_TU_WORKSPACE"    # ej: joss-workspace-7orjh
PROJECT   = "PEGA_AQUI_TU_PROYECTO"     # ej: giroscopio-dron
VERSION   = 1                            # número de versión del dataset

DATASET_LOCAL = '/content/dataset'

if 'DATASET_SOURCE' in dir() and DATASET_SOURCE == 'roboflow' or \
   not os.path.exists(os.path.join(DATASET_LOCAL, 'data.yaml')):
    from roboflow import Roboflow
    rf = Roboflow(api_key=API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    version = project.version(VERSION)
    dataset = version.download('yolov8', location=DATASET_LOCAL)
    print(f'\nDataset descargado en {DATASET_LOCAL}')
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(DATASET_LOCAL, split, 'images')
        n = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
        print(f'  {split}: {n} imágenes')
else:
    print('Dataset ya disponible en', DATASET_LOCAL, '— celda omitida.')


In [ ]:
# ============================================================
# CELDA 4 — Verificar y corregir el data.yaml
# ============================================================
# Roboflow a veces genera el data.yaml con rutas relativas
# que no funcionan en Colab. Aquí forzamos rutas absolutas
# y nos aseguramos de que kpt_shape sea [12, 3] (12 keypoints,
# cada uno con coordenadas x, y y visibilidad).
#
# También verificamos que las carpetas train/valid existen
# y tienen imágenes y labels, y comprobamos el formato de
# una anotación de muestra (debe tener 5 + 12×3 = 41 valores).

import yaml, os

DATASET_LOCAL = '/content/dataset'
yaml_path = os.path.join(DATASET_LOCAL, 'data.yaml')

with open(yaml_path) as f:
    data = yaml.safe_load(f)

print('=== data.yaml ORIGINAL ===')
print(yaml.dump(data, default_flow_style=False))

# Forzar rutas absolutas
data['path']  = DATASET_LOCAL
data['train'] = 'train/images'
data['val']   = 'valid/images'
data['test']  = 'test/images'

# 12 keypoints × 3 valores (x, y, visibilidad)
data['kpt_shape'] = [12, 3]

# Una sola clase
data['nc']    = 1
data['names'] = ['giroscopio']

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('=== data.yaml CORREGIDO ===')
with open(yaml_path) as f:
    print(f.read())

# Verificar que las rutas existen y tienen contenido
print('=== VERIFICACIÓN DE RUTAS ===')
for split in ['train', 'valid']:
    img_path = os.path.join(DATASET_LOCAL, split, 'images')
    lbl_path = os.path.join(DATASET_LOCAL, split, 'labels')
    n_img = len(os.listdir(img_path)) if os.path.exists(img_path) else 0
    n_lbl = len(os.listdir(lbl_path)) if os.path.exists(lbl_path) else 0
    status = '✓' if n_img > 0 and n_lbl > 0 else '⚠️'
    print(f'  {status} {split}: {n_img} imágenes, {n_lbl} labels')

# Comprobar formato de un label de muestra
# Formato esperado: 5 (bbox: clase, cx, cy, w, h) + 12×3 (x, y, v) = 41 valores
print('\n=== MUESTRA DE LABEL ===')
lbl_dir = os.path.join(DATASET_LOCAL, 'train', 'labels')
sample = sorted(os.listdir(lbl_dir))[0]
with open(os.path.join(lbl_dir, sample)) as f:
    line = f.readline().strip()
values = line.split()
N_KP = 12
EXPECTED = 5 + N_KP * 3  # = 41
print(f'  Archivo: {sample}')
print(f'  Valores: {len(values)} (esperados: {EXPECTED})')
if len(values) == EXPECTED:
    print(f'  ✓ Formato correcto')
else:
    print(f'  ⚠️  ERROR: se esperan {EXPECTED} valores, hay {len(values)}')
    print(f'     Comprueba que el dataset exportado de Roboflow tiene 12 keypoints')


In [ ]:
# ============================================================
# CELDA 5 — Entrenamiento
# ============================================================
# YOLOv8n-pose es la variante más ligera (nano) del modelo
# de pose de Ultralytics. Infiere automáticamente el número
# de keypoints desde el kpt_shape del data.yaml.
#
# Parámetros clave:
#   epochs   — número de pasadas completas sobre el dataset
#   imgsz    — resolución de entrada (640×640 px)
#   batch    — imágenes por paso de optimización (ajustar si
#              hay errores de memoria, bajar a 8)
#   patience — early stopping: para si no mejora en N épocas
#   project  — carpeta donde se guardan los resultados
#              (apuntamos directamente a Drive para no perder
#              nada si la sesión se reinicia)
#
# Los parámetros de augmentación (hsv_*, degrees, flip*,
# scale) añaden variaciones artificiales a las imágenes de
# entrenamiento para mejorar la generalización con datasets
# pequeños.

from ultralytics import YOLO
import torch, os

DRIVE_ROOT    = '/content/drive/MyDrive/giroscopio_proyecto_g1'
DATASET_LOCAL = '/content/dataset'
RUN_NAME      = 'giroscopio_v4'

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

model = YOLO('yolov8n-pose.pt')

results = model.train(
    data    = os.path.join(DATASET_LOCAL, 'data.yaml'),
    epochs  = 150,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    name    = RUN_NAME,
    project = os.path.join(DRIVE_ROOT, 'runs'),  # ← guardado directo en Drive
    # Augmentación
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    degrees = 15,
    flipud  = 0.3,
    fliplr  = 0.5,
    scale   = 0.5,
    patience= 30,
    save    = True,
    verbose = True,
)

print('\n=== RESULTADO DEL ENTRENAMIENTO ===')
print(f'mAP50    (pose): {results.results_dict["metrics/mAP50(P)"]:.4f}')
print(f'mAP50-95 (pose): {results.results_dict["metrics/mAP50-95(P)"]:.4f}')
print(f'mAP50    (box):  {results.results_dict["metrics/mAP50(B)"]:.4f}')
print(f'mAP50-95 (box):  {results.results_dict["metrics/mAP50-95(B)"]:.4f}')
print(f'Fitness combinado: {results.fitness:.4f}')
print(f'\nPesos guardados en: {DRIVE_ROOT}/runs/{RUN_NAME}/weights/')


In [ ]:
# ============================================================
# CELDA 6 — Verificar el modelo entrenado
# ============================================================
# Antes de exportar comprobamos que el modelo realmente
# aprendió a detectar el giroscopio:
#
#   1. Verificamos que tiene 12 keypoints en su cabeza
#   2. Imagen negra → no debe detectar nada (falsos positivos)
#   3. Imagen de ruido → confianza máxima < 0.3
#   4. Imagen real de validación → confianza > 0.3 y 12 KP
#
# Si alguna de las pruebas falla, revisar el dataset antes
# de continuar con la exportación.

import numpy as np
from ultralytics import YOLO
import os

DRIVE_ROOT    = '/content/drive/MyDrive/giroscopio_proyecto_g1'
RUN_NAME      = 'giroscopio_v4'
DATASET_LOCAL = '/content/dataset'

best_pt = os.path.join(DRIVE_ROOT, 'runs', RUN_NAME, 'weights', 'best.pt')
model   = YOLO(best_pt)

# 1. Comprobar arquitectura
kpt_shape = model.model.kpt_shape
print(f'kpt_shape del modelo: {kpt_shape}')
print('✓ 12 keypoints OK' if kpt_shape == [12, 3] else f'⚠️  Se esperaba [12, 3]')

# 2. Imagen negra (no debe detectar nada)
img_black = np.zeros((640, 640, 3), dtype=np.uint8)
r1 = model(img_black, verbose=False)
print(f'\nImagen negra    → detecciones: {len(r1[0].boxes)} (esperado: 0)')

# 3. Imagen de ruido (confianza baja)
img_noise = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
r2 = model(img_noise, verbose=False)
conf_noise = float(r2[0].boxes.conf.max()) if len(r2[0].boxes) else 0
print(f'Imagen ruido    → detecciones: {len(r2[0].boxes)}, conf máx: {conf_noise:.4f} (esperado: < 0.3)')

# 4. Imagen real de validación
import random
val_imgs = os.listdir(os.path.join(DATASET_LOCAL, 'valid', 'images'))
sample_img = os.path.join(DATASET_LOCAL, 'valid', 'images', random.choice(val_imgs))
r3 = model(sample_img, verbose=False)
conf_real = float(r3[0].boxes.conf.max()) if len(r3[0].boxes) else 0
print(f'Imagen real     → detecciones: {len(r3[0].boxes)}, conf máx: {conf_real:.4f} (esperado: > 0.3)')

if len(r3[0].boxes) > 0 and r3[0].keypoints is not None:
    n_kp = r3[0].keypoints.xy.shape[1]
    print(f'Keypoints/inst  → {n_kp} (esperado: 12)')
    print('✓' if n_kp == 12 else '⚠️  Número de keypoints incorrecto')

print()
if conf_real > 0.3:
    print('✅ El modelo detecta el giroscopio correctamente. Proceder con la exportación.')
else:
    print('⚠️  El modelo NO detecta bien. Revisar el dataset antes de exportar.')


In [ ]:
# ============================================================
# CELDA 7 — Exportar a ONNX (formato para Hailo)
# ============================================================
# IMPORTANTE: ejecutar solo si la Celda 6 confirmó que el
# modelo funciona correctamente.
#
# El pipeline de Hailo DFC (Data Flow Compiler) requiere un
# ONNX con salidas SEPARADAS (no un único tensor fusionado).
# Para ello necesitamos:
#   - opset 11 (máxima compatibilidad con Hailo DFC)
#   - simplify=True (elimina operaciones redundantes)
#   - dynamic=False (shapes fijas, requerido por Hailo)
#
# El modelo exportado y el .pt se guardan en la carpeta raíz
# del proyecto en Drive (giroscopio_proyecto_g1/).

from ultralytics import YOLO
import shutil, os

DRIVE_ROOT = '/content/drive/MyDrive/giroscopio_proyecto_g1'
RUN_NAME   = 'giroscopio_v4'

best_pt = os.path.join(DRIVE_ROOT, 'runs', RUN_NAME, 'weights', 'best.pt')
model   = YOLO(best_pt)

onnx_path = model.export(
    format   = 'onnx',
    imgsz    = 640,
    opset    = 11,
    simplify = True,
    dynamic  = False,
)
print(f'ONNX exportado en: {onnx_path}')

# Verificar el ONNX: con 12 KP y salidas separadas debe
# tener ≥ 9 tensores de salida (3 escalas × bbox + conf + kp)
# Si hay solo 1 salida fusionada, no sirve para Hailo.
import onnxruntime as ort
sess    = ort.InferenceSession(str(onnx_path))
outputs = sess.get_outputs()
print(f'\n=== VERIFICACIÓN DEL ONNX ===')
print(f'Número de salidas: {len(outputs)}')
for o in outputs:
    print(f'  {o.name}  shape={o.shape}')

if len(outputs) == 1:
    print('\n⚠️  ONNX con salida fusionada — NO sirve para Hailo')
    print('   Solución: !pip install ultralytics --upgrade -q  y reexportar')
elif len(outputs) >= 9:
    print('\n✅ ONNX con salidas separadas — correcto para Hailo')
else:
    print(f'\n? ONNX con {len(outputs)} salidas — verificar manualmente')

# Copiar los archivos finales a la carpeta raíz del proyecto
dest_onnx = os.path.join(DRIVE_ROOT, f'giroscopio_pose_best_n_{RUN_NAME}.onnx')
dest_pt   = os.path.join(DRIVE_ROOT, f'giroscopio_pose_best_n_{RUN_NAME}.pt')

shutil.copy(str(onnx_path), dest_onnx)
shutil.copy(best_pt, dest_pt)

print(f'\nArchivos guardados en {DRIVE_ROOT}:')
print(f'  {os.path.basename(dest_onnx)}')
print(f'  {os.path.basename(dest_pt)}')
print('\n✅ Proceso completo.')
